### 랭체인 준비

In [55]:
# 패키지 설치
!pip install langchain
!pip install langchain-google-genai
!pip install langchain_community
!pip install langgraph

In [56]:
import os
from google.colab import userdata

# 환경 변수 준비(좌측 상단의 열쇠 아이콘으로 GOOGLE_API_KEY 설정)
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

### LLM 준비

In [57]:
from langchain_google_genai import ChatGoogleGenerativeAI

# LLM 준비
llm = ChatGoogleGenerativeAI(
    model="models/gemini-1.5-flash",
)

In [58]:
# 대화
llm.invoke([("human", "안녕하세요! 제 이름은 승민입니다.")])

AIMessage(content='안녕하세요, 승민님! 만나서 반가워요. 무엇을 도와드릴까요? \n', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}]}, id='run-df0971c9-3402-434c-a7ed-054fe2196759-0', usage_metadata={'input_tokens': 12, 'output_tokens': 26, 'total_tokens': 38, 'input_token_details': {'cache_read': 0}})

In [59]:
# 과거 대화 내용에 관해 질문
llm.invoke([("human", "내 이름은?")])

AIMessage(content='저는 구글에서 훈련된 대규모 언어 모델입니다. 이름은 없어요. \n\n무엇을 도와드릴까요? \n', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}]}, id='run-6fadaec3-79ca-41ab-90c5-62314fb7684b-0', usage_metadata={'input_tokens': 5, 'output_tokens': 35, 'total_tokens': 40, 'input_token_details': {'cache_read': 0}})

In [60]:
from langchain_core.messages import AIMessage

# 대화 이력과 함께 과거 대화 내용에 관해 질문
llm.invoke(
    [
        ("human", "안녕하세요! 제 이름은 승민입니다."),
        ("ai", "안녕하세요, 승민! 만나서 반갑습니다. 오늘은 어떤 도움이 필요하신가요?"),
        ("human", "내 이름은?"),
    ]
)

AIMessage(content='승민님, 당신의 이름은 **승민**입니다! \n\n혹시 다른 질문이 있으신가요? \n', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}]}, id='run-3e749db3-9e29-4225-b207-d1500ca993d3-0', usage_metadata={'input_tokens': 46, 'output_tokens': 28, 'total_tokens': 74, 'input_token_details': {'cache_read': 0}})

### ChatBot 준비

In [61]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# 대화 이력을 저장할 공간을 준비
store = {}

# 세션마다 대화 이력을 가져오기
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# ChatBot 준비
chatbot = RunnableWithMessageHistory(llm, get_session_history)

In [62]:
# 설정 준비
config = {"configurable": {"session_id": "abc1"}}

In [63]:
# 대화
response = chatbot.invoke(
    [("human", "안녕하세요! 제 이름은 승민입니다.")],
    config=config,
)
response.content

'안녕하세요, 승민님! 만나서 반가워요! 무엇을 도와드릴까요? \n'

In [64]:
# 과거 대화 내용에 관해 질문
response = chatbot.invoke(
    [("human", "내 이름은?")],
    config=config,
)
response.content

'승민님, 혹시 잊으셨나요? 당신의 이름은 **승민**입니다! \n'

### 커스텀 지시

In [77]:
from langchain_core.prompts import ChatPromptTemplate

# PromptTemplate 준비
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 훌륭한 AI비서입니다. 말 끝에 '~이다.'를 붙여 대답해주세요."),
        ("placeholder", "{messages}"),
    ]
)

# Chain 준비
chain = prompt_template | llm

In [78]:
# ChatBot 준비
chatbot = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)

In [79]:
# 설정 준비
config = {"configurable": {"session_id": "abc2"}}

In [80]:
# 대화
response = chatbot.invoke(
    {"messages": [("human", "안녕하세요! 제 이름은 승민입니다.")]},
    config=config
)
response.content

'안녕하세요! 승민님, 만나서 반가워요. 😄 \n'

In [81]:
# 과거 대화 내용에 관해 질문
response = chatbot.invoke(
    {"messages": [("human", "내 이름은?")]},
    config=config,
)
response.content

'승민님의 이름은 승민입니다. 😊 \n'

### 대화 이력 관리

In [82]:
from langchain_core.runnables import RunnablePassthrough

# 메시지 필터 준비
def filter_messages(messages, k=4):
    return messages[-k:]

# Chain 준비
chain = (
    RunnablePassthrough.assign(messages=lambda x: filter_messages(x["messages"]))
    | prompt_template
    | llm
)

In [83]:
# ChatBot 준비
chatbot = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)

In [84]:
# 설정 준비
config = {"configurable": {"session_id": "abc3"}}

In [85]:
# 대화
response = chatbot.invoke(
    {"messages": [("human", "안녕하세요! 제 이름은 승민입니다.")]},
    config=config,
)
response.content

'안녕하세요, 승민님! 반가워요. 😄 \n'

In [86]:
# 과거 대화 내용에 관해 질문
response = chatbot.invoke(
    {"messages": [("human", "내 이름은?")]},
    config=config,
)
response.content

'승민님의 이름은 승민입니다. 😊 \n'

In [87]:
# 과거 대화 내용에 관해 질문(2회차)
response = chatbot.invoke(
    {"messages": [("human", "내 이름은?")]},
    config=config,
)
response.content

'승민님의 이름은 승민입니다. 😊 \n'

### 랭스미스

In [76]:
import os
from uuid import uuid4

# 환경 변수 준비
unique_id = uuid4().hex[0:8]
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"Tracing Walkthrough - {unique_id}"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")